In [4]:
import boto3
import csv
import json
import time
import random

# ---- Config ----
STREAM_NAME = "ecomm-firehose-str"
CSV_FILE = "2026-Jun-sample.csv"
REGION = "us-east-1"          # apni region daal do
PROFILE = None                 # agar named profile use kar rahe ho to naam daal do, warna default rehne do
BATCH_SIZE = 20                 # ek dafa mein kitni rows bhejni hain (Firehose PutRecordBatch max 500 leta hai)
DELAY_SECONDS = 2                # har batch ke baad kitna wait karna hai (real-time feel ke liye)

# ---- Boto3 client setup ----
session = boto3.Session(profile_name=PROFILE, region_name=REGION) if PROFILE else boto3.Session(region_name=REGION)
firehose_client = session.client("firehose")

def send_batch(records):
    """Records ko Firehose ke required format mein convert karke batch mein bhejta hai"""
    entries = [{"Data": (json.dumps(record) + "\n").encode("utf-8")} for record in records]
    response = firehose_client.put_record_batch(
        DeliveryStreamName=STREAM_NAME,
        Records=entries
    )
    failed = response.get("FailedPutCount", 0)
    if failed > 0:
        print(f"⚠️  {failed} records fail hue is batch mein")
    else:
        print(f"✅ {len(entries)} records successfully bhej diye")

def main():
    with open(CSV_FILE, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        batch = []
        total_sent = 0

        for row in reader:
            batch.append(row)

            if len(batch) >= BATCH_SIZE:
                send_batch(batch)
                total_sent += len(batch)
                batch = []
                # thora random delay taake bilkul mechanical na lage
                time.sleep(DELAY_SECONDS + random.uniform(-0.5, 0.5))

        # last leftover batch bhi bhej do
        if batch:
            send_batch(batch)
            total_sent += len(batch)

        print(f"\n🎉 Total {total_sent} records Firehose ko bhej diye gaye")

if __name__ == "__main__":
    main()

✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bhej diye
✅ 20 records successfully bh

KeyboardInterrupt: 